# Lesson 08 Walkthrough: Natural Language Processing
**Artificial Intelligence — Medina County Career Center**

Computers don't understand words — they understand **numbers**. Natural Language Processing (NLP) is the set of tricks we use to turn text into numbers a model can learn from, and to turn model output back into text. Today we do two classic NLP tasks:

1. **Sentiment analysis** — teach a model to tell positive reviews from negative ones.
2. **Text generation** — build a tiny model that writes new sentences.

Run each cell in order (Shift+Enter).

## Part 1 — A sentiment dataset

First we need labeled examples: reviews tagged **positive (1)** or **negative (0)**. We generate them so the notebook is self-contained and everyone gets identical results.

In [ ]:
import random
import pandas as pd

# We GENERATE a labeled review dataset so the lesson is self-contained and
# reproducible (Random(42) => everyone gets the same data & the same results).
pos_words = ["great","amazing","wonderful","excellent","fantastic","brilliant",
             "loved","enjoyable","superb","delightful","outstanding","charming"]
neg_words = ["terrible","awful","boring","horrible","dreadful","bad",
             "hated","dull","disappointing","tedious","forgettable","clumsy"]
templates = ["a {} and {} movie","the acting was {} and {}",
             "i thought it was {} and truly {}","what a {} {} film",
             "{} story with {} characters","an {} and {} experience"]
rng = random.Random(42)
def make(words, n):
    return [rng.choice(templates).format(rng.choice(words), rng.choice(words))
            for _ in range(n)]
pos = make(pos_words, 90)
neg = make(neg_words, 90)
# a few AMBIGUOUS reviews (mostly one sentiment, one opposite word) so the
# model isn't perfect -- these are the ones it tends to get wrong.
mixed_pos = [f"a {rng.choice(pos_words)} film but a bit {rng.choice(neg_words)}" for _ in range(8)]
mixed_neg = [f"a {rng.choice(neg_words)} film but a bit {rng.choice(pos_words)}" for _ in range(8)]
reviews = pos + mixed_pos + neg + mixed_neg
labels  = [1]*98 + [0]*98            # 1 = positive, 0 = negative
df = pd.DataFrame({"review": reviews, "sentiment": labels})

print("rows:", len(df))                     # Should print 196
print("positive:", int(df.sentiment.sum()))  # Should print 98
print(df.sample(3, random_state=1).to_string(index=False))

## Part 2 — Turning text into numbers (Bag-of-Words)

You met **Bag-of-Words** in the ai04 cert insert. `CountVectorizer` builds a vocabulary of every word, then represents each review as a row of word **counts**. The order of words is thrown away — just the counts remain. That's the "bag."

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
example = vectorizer.fit_transform(["a great and amazing movie", "a boring movie"])
print("vocabulary:", vectorizer.get_feature_names_out().tolist())
print("count matrix:")
print(example.toarray())   # one row per review, one column per word

## Part 3 — Train a sentiment classifier

Same recipe you already know: **split**, **vectorize**, **train**, **evaluate**. The model is `LogisticRegression` — a fast, reliable classifier. We fit the vectorizer on the **training** text only, then apply it to the test text.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['sentiment'], test_size=0.25, random_state=42, stratify=df['sentiment'])
print('train/test sizes:', len(X_train), len(X_test))   # Should print 147 49

vec = CountVectorizer()
X_train_vec = vec.fit_transform(X_train)   # LEARN vocab on training text
X_test_vec  = vec.transform(X_test)        # APPLY same vocab to test text
print('vocabulary size:', len(vec.vocabulary_))          # Should print 41

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)
pred = model.predict(X_test_vec)
print('accuracy:', round(accuracy_score(y_test, pred), 3))       # Should print 0.939
print('confusion:', confusion_matrix(y_test, pred).tolist())      # Should print [[24, 1], [2, 22]]

## Part 4 — Read the results

`confusion` printed `[[24, 1], [2, 22]]` = `[[TN, FP], [FN, TP]]`. Only **3** wrong out of 49. The classification report shows precision/recall per class — the metrics you learned in ai04.

In [ ]:
print(classification_report(y_test, pred, target_names=['negative','positive']))

The handful it misses are the **ambiguous** reviews — the "a great film but a bit boring" ones. Mixed sentiment and negation are genuinely hard for Bag-of-Words, because it just counts words and can't tell *"not good"* from *"good."*

## Part 5 — Predict on brand-new sentences

The real payoff: run the trained model on text it has never seen.

In [ ]:
new_reviews = ['a wonderful and charming movie', 'a boring and awful film']
print('predictions:', model.predict(vec.transform(new_reviews)).tolist())  # Should print [1, 0]
# 1 = positive, 0 = negative

## Part 6 — Text generation with a Markov chain

Now we flip NLP around: instead of reading text, we **write** it. The simplest generator is a **Markov chain**: look at the current word, and pick the next word from the words that actually followed it in the training text. No neural network, no training loop — just counting what comes after what.

In [ ]:
# A small "training text" for the generator. Any text works -- swap this out!
corpus_text = """the movie was a thrilling ride from start to finish .
the story was slow but the acting was strong .
a bold and clever film with a satisfying ending .
the plot was thin but the visuals were stunning .
a fun adventure with heart and a great cast .
the pacing was off but the music was beautiful .
a smart and funny film that keeps you guessing .
the ending was weak but the journey was worth it ."""
words = corpus_text.split()

def build_markov(words, order=1):
    """Map each key (a tuple of `order` words) -> list of words that followed it."""
    chain = {}
    for i in range(len(words) - order):
        key = tuple(words[i:i+order])
        chain.setdefault(key, []).append(words[i+order])
    return chain

def generate(chain, start_key, n, seed=0):
    import random
    random.seed(seed)
    order = len(start_key)
    out = list(start_key)
    key = start_key
    for _ in range(n - order):
        choices = chain.get(key)
        if not choices:
            break
        out.append(random.choice(choices))
        key = tuple(out[-order:])
    return " ".join(out)

chain1 = build_markov(words, order=1)
print('states:', len(chain1))                              # Should print 47
print(generate(chain1, ('the',), 12, seed=1))   # Should print the acting was off but the ending was weak but the ending

Change the `seed` and you get a different sentence from the same chain. It's grammatical-ish but wanders, because order-1 only remembers **one** word back.

## Part 7 — Order-2: more memory, more coherence

Give the chain **two** words of context and it sounds much more like the source — but it also starts to just replay training phrases. That trade-off (coherence vs. originality) is the whole story of language models, all the way up to ChatGPT.

In [ ]:
chain2 = build_markov(words, order=2)
print('states:', len(chain2))                              # Should print 72
print(generate(chain2, ('the','story'), 12, seed=3))   # Should print the story was slow but the acting was strong . a bold

## The grown-up version: RNNs and Transformers

A Markov chain only looks back 1–2 words. Real language models — **RNNs/LSTMs** (what CodeHS uses) and **Transformers** (ChatGPT, Gemini) — look back over hundreds of words and learn the patterns instead of memorizing counts. Same core idea you just built: **predict the next token from the ones before it.** You now understand the engine underneath the world's biggest AI systems.

**On to the tasks!**